In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import json
import os
import time
from dotenv import load_dotenv

load_dotenv()
service_key = os.getenv("MOLTI_API_KEY")

# 시군구 매핑 로드
with open("./Data/SigunguMapping.json", "r") as f:
    sigungu_data = json.load(f)

# 수도권 시군구코드 수집
capital_codes = []
for sido in ["서울특별시", "경기도", "인천시"]:
    for code in sigungu_data[sido].keys():
        capital_codes.append(code)

# 원하는 기간 설정 (예: 2022년 1월 ~ 2024년 12월)
year_months = []
for year in range(2022, 2025):
    for month in range(1, 13):
        year_months.append(f"{year}{month:02d}")

# API 호출 함수
def fetch_apt_trade(sigungu_code, deal_ymd):
    url = "http://apis.data.go.kr/1613000/RTMSDataSvcAptTradeDev/getRTMSDataSvcAptTradeDev"
    params = {
        "serviceKey": service_key,
        "LAWD_CD": sigungu_code,
        "DEAL_YMD": deal_ymd,
        "numOfRows": "9999",
        "pageNo": "1",
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return pd.DataFrame()
    
    root = ET.fromstring(response.text)
    items = root.findall(".//item")
    
    data_list = []
    for item in items:
        row = {}
        for child in item:
            row[child.tag] = child.text
        data_list.append(row)
    
    return pd.DataFrame(data_list)

# 전체 수집
all_data = []
for code in capital_codes:
    for ym in year_months:
        print(f"수집 중: {code} - {ym}")
        df = fetch_apt_trade(code, ym)
        if not df.empty:
            all_data.append(df)
        time.sleep(0.5)  # API 과부하 방지

result = pd.concat(all_data, ignore_index=True)
print(f"총 {len(result)}건 수집 완료")
result.to_csv("./Data/apt_trade_capital.csv", index=False, encoding="utf-8-sig")

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import json
import os
import time
from dotenv import load_dotenv

load_dotenv()
service_key = os.getenv("MOLTI_API_KEY")

# 샘플 1건 테스트 (서초구, 2022년 12월)

# API 호출 함수
def fetch_apt_trade(sigungu_code, deal_ymd):
    url = "http://apis.data.go.kr/1613000/RTMSDataSvcAptTradeDev/getRTMSDataSvcAptTradeDev"
    params = {
        "serviceKey": service_key,
        "LAWD_CD": sigungu_code,
        "DEAL_YMD": deal_ymd,
        "numOfRows": "9999",
        "pageNo": "1",
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return pd.DataFrame()
    
    root = ET.fromstring(response.text)
    items = root.findall(".//item")
    
    data_list = []
    for item in items:
        row = {}
        for child in item:
            row[child.tag] = child.text
        data_list.append(row)
    
    return pd.DataFrame(data_list)

sample_df = fetch_apt_trade("11650", "202212")
print(f"총 {len(sample_df)}건")
sample_df.head(1)

In [ ]:
sample_df.columns

In [ ]:
import pandas as pd
import openpyxl

In [ ]:
maemae_2005_2011 = pd.read_excel("./Data/매매/2005~2011.xlsx")
maemae_2012_2026 = pd.read_excel("./Data/매매/2012~2026.xlsx")


In [ ]:
maemae_2005_2011.columns

In [ ]:
maemae_2012_2026.columns

In [ ]:
# 2012~2026을 먼저, 2005~2011을 뒤에
maemae_df = pd.concat([maemae_2012_2026, maemae_2005_2011], ignore_index=True)
maemae_df.columns



In [ ]:
maemae_df['NO'] = range(1, len(maemae_df) + 1)
maemae_df.head()

In [ ]:
maemae_df["NO"]

In [ ]:
# maemae_df.to_excel("./Data/Trade_Data.xlsx", index=False, engine='openpyxl')
maemae_df.to_csv("./Data/Trade_Data.csv", index=False, encoding="utf-8-sig")

In [ ]:
jeonse_2011 = pd.read_excel("./Data/전세_월세/2011.xlsx")
jeonse_2012_2017 = pd.read_excel("./Data/전세_월세/2012~2017.xlsx")
jeonse_2018_2022 = pd.read_excel("./Data/전세_월세/2018~2022.xlsx")
jeonse_2023_2026 = pd.read_excel("./Data/전세_월세/2023~2026.xlsx")

In [ ]:
total_jeonse = pd.concat([jeonse_2023_2026, jeonse_2018_2022, jeonse_2012_2017, jeonse_2011], ignore_index=True)
total_jeonse.columns
total_jeonse['NO'] = range(1, len(total_jeonse) + 1)
total_jeonse.head()



In [ ]:
jeonse_df = total_jeonse[total_jeonse["전월세구분"] == "전세"]
wolse_df = total_jeonse[total_jeonse["전월세구분"] == "월세"]
jeonse_df["전월세구분"].unique()

In [ ]:
jeonse_df.to_csv("./Data/Jeonse_Data.csv", index=False, encoding="utf-8-sig")
wolse_df.to_csv("./Data/Rent_Data.csv", index=False, encoding="utf-8-sig")

In [ ]:
jeonse_df['NO'] = range(1, len(jeonse_df) + 1)
wolse_df['NO'] = range(1, len(wolse_df) + 1)
jeonse_df['NO']
wolse_df['NO']

In [ ]:
jeonse_df = jeonse_df.reset_index(drop=True)

# NO 컬럼을 1부터 시작하는 연속된 번호로 재설정
jeonse_df['NO'] = range(1, len(jeonse_df) + 1)
jeonse_df['NO']

In [ ]:
wolse_df = wolse_df.reset_index(drop=True)

# NO 컬럼을 1부터 시작하는 연속된 번호로 재설정
wolse_df['NO'] = range(1, len(wolse_df) + 1)
wolse_df['NO']

In [ ]:
jeonse_df.to_csv("./Data/Jeonse_Data.csv", index=False, encoding="utf-8-sig")
wolse_df.to_csv("./Data/Rent_Data.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
Rent_df = pd.read_csv("./Data/Rent_Data.csv")
Jeonse_df = pd.read_csv("./Data/Jeonse_Data.csv")
Trade_df = pd.read_csv("./Data/Trade_Data.csv")

In [4]:
Trade_df.columns

Index(['NO', '시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)', '계약년월', '계약일',
       '거래금액(만원)', '동', '층', '매수자', '매도자', '건축년도', '도로명', '해제사유발생일', '거래유형',
       '중개사소재지', '등기일자', '주택유형'],
      dtype='str')

In [5]:
Jeonse_df.columns

Index(['NO', '시군구', '번지', '본번', '부번', '단지명', '전월세구분', '전용면적(㎡)', '계약년월', '계약일',
       '보증금(만원)', '월세금(만원)', '층', '건축년도', '도로명', '계약기간', '계약구분', '갱신요구권 사용',
       '종전계약 보증금(만원)', '종전계약 월세(만원)', '주택유형'],
      dtype='str')

In [6]:
Rent_df.columns

Index(['NO', '시군구', '번지', '본번', '부번', '단지명', '전월세구분', '전용면적(㎡)', '계약년월', '계약일',
       '보증금(만원)', '월세금(만원)', '층', '건축년도', '도로명', '계약기간', '계약구분', '갱신요구권 사용',
       '종전계약 보증금(만원)', '종전계약 월세(만원)', '주택유형'],
      dtype='str')

In [7]:
# 세 데이터프레임의 공통 컬럼 찾기
common_cols = set(Trade_df.columns) & set(Jeonse_df.columns) & set(Rent_df.columns)
print("공통 컬럼:")
print(sorted(common_cols))

공통 컬럼:
['NO', '건축년도', '계약년월', '계약일', '단지명', '도로명', '번지', '본번', '부번', '시군구', '전용면적(㎡)', '주택유형', '층']


In [10]:
# 원하는 컬럼들만 선택
cols_to_compare = ['건축년도', '계약년월', '계약일', '단지명', '도로명', '번지', '본번', '부번', '시군구', '전용면적(㎡)', '주택유형', '층']

print("=== Trade_df ===")
print(Trade_df[cols_to_compare].head())

print("\n=== Jeonse_df ===")
print(Jeonse_df[cols_to_compare].head())

print("\n=== Rent_df ===")
print(Rent_df[cols_to_compare].head())

=== Trade_df ===
   건축년도    계약년월  계약일                        단지명          도로명      번지   본번  부번  \
0  2022  202602   13                      테라파크뷰  동일로192다길 51  366-15  366  15   
1  2002  202602   13                     경남아너스빌    언주로85길 13     722  722   0   
2  2012  202602   13                 래미안 옥수 리버젠       매봉길 15     561  561   0   
3  2009  202602   13  우물골2단지두산위브(245~255동)BL2-6    진관2로 57-7      81   81   0   
4  1993  202602   13                 대림아파트(276)      통일로 586     276  276   0   

             시군구  전용면적(㎡) 주택유형   층  
0  서울특별시 노원구 공릉동   55.200  아파트  10  
1  서울특별시 강남구 역삼동   84.688  아파트  13  
2  서울특별시 성동구 옥수동  134.130  아파트   7  
3  서울특별시 은평구 진관동   84.710  아파트   6  
4  서울특별시 은평구 녹번동   84.940  아파트  15  

=== Jeonse_df ===
   건축년도    계약년월  계약일                    단지명         도로명      번지    본번  부번  \
0  1988  202602   13       상계주공15(고층,공무원임대)  동일로227길 26     624   624   0   
1  1998  202602   13  한신아파트상가동유치원동(103~109)     동일로 752     450   450   0   
2  2014  202602   13    

In [11]:
import pandas as pd

# 컬럼 비교 데이터프레임 만들기
comparison = pd.DataFrame({
    'Trade_df': Trade_df.columns.tolist(),
    'Jeonse_df': Jeonse_df.columns.tolist(),
    'Rent_df': Rent_df.columns.tolist()
})

print(comparison)

    Trade_df     Jeonse_df       Rent_df
0         NO            NO            NO
1        시군구           시군구           시군구
2         번지            번지            번지
3         본번            본번            본번
4         부번            부번            부번
5        단지명           단지명           단지명
6    전용면적(㎡)         전월세구분         전월세구분
7       계약년월       전용면적(㎡)       전용면적(㎡)
8        계약일          계약년월          계약년월
9   거래금액(만원)           계약일           계약일
10         동       보증금(만원)       보증금(만원)
11         층       월세금(만원)       월세금(만원)
12       매수자             층             층
13       매도자          건축년도          건축년도
14      건축년도           도로명           도로명
15       도로명          계약기간          계약기간
16   해제사유발생일          계약구분          계약구분
17      거래유형      갱신요구권 사용      갱신요구권 사용
18    중개사소재지  종전계약 보증금(만원)  종전계약 보증금(만원)
19      등기일자   종전계약 월세(만원)   종전계약 월세(만원)
20      주택유형          주택유형          주택유형


In [12]:
# NO 컬럼 제외
trade_cols = [col for col in Trade_df.columns if col != 'NO']
jeonse_cols = [col for col in Jeonse_df.columns if col != 'NO']
rent_cols = [col for col in Rent_df.columns if col != 'NO']

# 컬럼 비교 데이터프레임 만들기
comparison = pd.DataFrame({
    'Trade_df': trade_cols,
    'Jeonse_df': jeonse_cols,
    'Rent_df': rent_cols
})

print(comparison)

    Trade_df     Jeonse_df       Rent_df
0        시군구           시군구           시군구
1         번지            번지            번지
2         본번            본번            본번
3         부번            부번            부번
4        단지명           단지명           단지명
5    전용면적(㎡)         전월세구분         전월세구분
6       계약년월       전용면적(㎡)       전용면적(㎡)
7        계약일          계약년월          계약년월
8   거래금액(만원)           계약일           계약일
9          동       보증금(만원)       보증금(만원)
10         층       월세금(만원)       월세금(만원)
11       매수자             층             층
12       매도자          건축년도          건축년도
13      건축년도           도로명           도로명
14       도로명          계약기간          계약기간
15   해제사유발생일          계약구분          계약구분
16      거래유형      갱신요구권 사용      갱신요구권 사용
17    중개사소재지  종전계약 보증금(만원)  종전계약 보증금(만원)
18      등기일자   종전계약 월세(만원)   종전계약 월세(만원)
19      주택유형          주택유형          주택유형


In [13]:
# NO 제외하고 공통 컬럼 찾기
trade_cols = set([col for col in Trade_df.columns if col != 'NO'])
jeonse_cols = set([col for col in Jeonse_df.columns if col != 'NO'])
rent_cols = set([col for col in Rent_df.columns if col != 'NO'])

common_cols = list(trade_cols & jeonse_cols & rent_cols)

# 공통 컬럼만 선택해서 합치기
trade_selected = Trade_df[common_cols].copy()
jeonse_selected = Jeonse_df[common_cols].copy()
rent_selected = Rent_df[common_cols].copy()

# 데이터 타입 구분을 위한 컬럼 추가
trade_selected['거래유형'] = '매매'
jeonse_selected['거래유형'] = '전세'
rent_selected['거래유형'] = '월세'

# 세 데이터프레임 합치기
combined_df = pd.concat([trade_selected, jeonse_selected, rent_selected], ignore_index=True)

print(f"합쳐진 데이터프레임 크기: {combined_df.shape}")
print(f"컬럼: {list(combined_df.columns)}")
print(combined_df.head())

합쳐진 데이터프레임 크기: (4341903, 13)
컬럼: ['본번', '시군구', '계약년월', '건축년도', '번지', '주택유형', '도로명', '계약일', '층', '전용면적(㎡)', '부번', '단지명', '거래유형']
    본번            시군구    계약년월  건축년도      번지 주택유형          도로명  계약일     층  \
0  366  서울특별시 노원구 공릉동  202602  2022  366-15  아파트  동일로192다길 51   13  10.0   
1  722  서울특별시 강남구 역삼동  202602  2002     722  아파트    언주로85길 13   13  13.0   
2  561  서울특별시 성동구 옥수동  202602  2012     561  아파트       매봉길 15   13   7.0   
3   81  서울특별시 은평구 진관동  202602  2009      81  아파트    진관2로 57-7   13   6.0   
4  276  서울특별시 은평구 녹번동  202602  1993     276  아파트      통일로 586   13  15.0   

   전용면적(㎡)  부번                        단지명 거래유형  
0   55.200  15                      테라파크뷰   매매  
1   84.688   0                     경남아너스빌   매매  
2  134.130   0                 래미안 옥수 리버젠   매매  
3   84.710   0  우물골2단지두산위브(245~255동)BL2-6   매매  
4   84.940   0                 대림아파트(276)   매매  


In [17]:
combined_df

,본번,시군구,계약년월,건축년도,번지,주택유형,도로명,계약일,층,전용면적(㎡),부번,단지명,거래유형
0,366,서울특별시 노원구 공릉동,202602,2022,366-15,아파트,동일로192다길 51,13,10.0,55.200,15,테라파크뷰,매매
1,722,서울특별시 강남구 역삼동,202602,2002,722,아파트,언주로85길 13,13,13.0,84.688,0,경남아너스빌,매매
2,561,서울특별시 성동구 옥수동,202602,2012,561,아파트,매봉길 15,13,7.0,134.130,0,래미안 옥수 리버젠,매매
3,81,서울특별시 은평구 진관동,202602,2009,81,아파트,진관2로 57-7,13,6.0,84.710,0,우물골2단지두산위브(245~255동)BL2-6,매매
4,276,서울특별시 은평구 녹번동,202602,1993,276,아파트,통일로 586,13,15.0,84.940,0,대림아파트(276),매매
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4341898,414,서울특별시 강동구 길동,201101,2002,414-6,아파트,천호대로177길 55,1,10.0,56.700,6,신암가온,월세
4341899,1022,서울특별시 성북구 정릉동,201101,2003,1022,아파트,정릉로 185-4,1,11.0,114.890,0,대동아파트,월세
4341900,930,서울특별시 양천구 목동,201101,1996,930,아파트,목동동로 202,1,17.0,83.160,0,현대,월세
4341901,1276,서울특별시 양천구 신정동,201101,2000,1276,아파트,신정로 290,1,6.0,84.970,0,신트리3,월세


In [ ]:
# 공통 컬럼 찾기 (NO 제외)
trade_cols = set([col for col in Trade_df.columns if col != 'NO'])
jeonse_cols = set([col for col in Jeonse_df.columns if col != 'NO'])
common_cols = list(trade_cols & jeonse_cols)

print(f"공통 컬럼: {common_cols}")

# 매칭에 사용할 키 컬럼 선택 (같은 물건을 식별할 컬럼들)
key_cols = ['시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)', '층', '건축년도']

# 실제로 두 데이터프레임에 모두 존재하는 키 컬럼만 선택
key_cols = [col for col in key_cols if col in common_cols]

print(f"\n매칭 키 컬럼: {key_cols}")

# 매매 데이터 중 전세 데이터와 매칭되는 것만 필터링
trade_matched = Trade_df.merge(
    Jeonse_df[key_cols].drop_duplicates(),
    on=key_cols,
    how='inner'
)

print(f"\n매매 전체 데이터: {len(Trade_df):,}개")
print(f"전세와 매칭되는 매매 데이터: {len(trade_matched):,}개")
print(f"매칭 비율: {len(trade_matched)/len(Trade_df)*100:.2f}%")

# 결과 확인
print("\n매칭된 데이터 샘플:")
print(trade_matched[key_cols + ['거래금액(만원)']].head(10))

공통 컬럼: ['전용면적(㎡)', '본번', '계약년월', '건축년도', '번지', '단지명', '주택유형', '도로명', '층', '시군구', '부번', '계약일']

매칭 키 컬럼: ['시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)', '계약년월', '계약일', '층', '건축년도']

매매 전체 데이터: 1,394,160개
전세와 매칭되는 매매 데이터: 17,211개
매칭 비율: 1.23%

매칭된 데이터 샘플:
               시군구     번지    본번  부번         단지명   전용면적(㎡)    계약년월  계약일   층  \
0    서울특별시 노원구 상계동    720   720   0   상계주공6(고층)   58.0100  202602    7  10   
1    서울특별시 송파구 문정동    145   145   0        문정시영   39.6900  202602    7  10   
2  서울특별시 성동구 금호동4가    800   800   0          대우   84.7100  202601   31  19   
3    서울특별시 양천구 신정동   1273  1273   0       신정현대5  114.2400  202601   31   7   
4    서울특별시 강서구 마곡동    744   744   0    마곡엠밸리9단지   59.8600  202601   31  14   
5    서울특별시 성북구 돈암동    636   636   0   이수브라운스톤돈암   59.9900  202601   31  18   
6   서울특별시 중구 인현동2가    240   240   0  세운푸르지오헤리시티   24.5533  202601   31  12   
7    서울특별시 노원구 상계동    639   639   0       보람아파트   68.9900  202601   30   4   
8    서울특별시 노원구 중계동  362-1   362   1         주공8 

In [20]:
trade_matched["계약년월"]

0        202602
1        202602
2        202601
3        202601
4        202601
          ...  
17206    201101
17207    201101
17208    201101
17209    201101
17210    201101
Name: 계약년월, Length: 17211, dtype: int64

In [22]:
trade_matched["계약년월"].unique()

array([202602, 202601, 202512, 202511, 202510, 202509, 202508, 202507,
       202506, 202505, 202504, 202503, 202502, 202501, 202412, 202411,
       202410, 202409, 202408, 202407, 202406, 202405, 202404, 202403,
       202402, 202401, 202312, 202311, 202310, 202309, 202308, 202307,
       202306, 202305, 202304, 202303, 202302, 202301, 202212, 202211,
       202210, 202209, 202208, 202207, 202206, 202205, 202204, 202203,
       202202, 202201, 202112, 202111, 202110, 202109, 202108, 202107,
       202106, 202105, 202104, 202103, 202102, 202101, 202012, 202011,
       202010, 202009, 202008, 202007, 202006, 202005, 202004, 202003,
       202002, 202001, 201912, 201911, 201910, 201909, 201908, 201907,
       201906, 201905, 201904, 201903, 201902, 201901, 201812, 201811,
       201810, 201809, 201808, 201807, 201806, 201805, 201804, 201803,
       201802, 201801, 201712, 201711, 201710, 201709, 201708, 201707,
       201706, 201705, 201704, 201703, 201702, 201701, 201612, 201611,
      

In [25]:
Jeonse_df["계약기간"]

0          202604~202804
1          202602~202802
2          202604~202804
3          202605~202805
4          202604~202804
               ...      
1934463                -
1934464                -
1934465                -
1934466                -
1934467                -
Name: 계약기간, Length: 1934468, dtype: str

전세 계약기간이 현재까지 남ㅇ아있다면 현재까지 갭투자비율
매매했을때 전세 계약기간안에 포함되어있으면 매매계약이 실거주인지 갭투자인지 알수있음

이것들을 함수로 만들면 연도별 지역별 갭투자 비율을 만들수있음음

In [ ]:
# 